# 🎯 Code-Generation Uncertainty Quantification

<div style="background-color: rgba(200, 200, 200, 0.1); padding: 20px; border-radius: 8px; margin-bottom: 20px; border: 1px solid rgba(127, 127, 127, 0.2); max-width: 97.5%; overflow-wrap: break-word;">
  <p style="font-size: 16px; line-height: 1.6">
    Code-Generation Uncertainty Quantification (UQ) methods estimate response-level confidence scores for coding tasks. This demo provides an illustration 
    of how to use state-of-the-art UQ methods with <code>uqlm</code>. Here is the list of available scores:
  </p>
      
*   Black-Box UQ ["functional_entropy", "semantic_sets", "cosine_sim"]
*   White-Box UQ ["sequence_probability", "min_probability", "mean_token_negentropy", "min_token_negentropy", "probability_margin", "p_true", "monte_carlo_probability"]
*   Judges ["codebleu", "code_equivalence", 
         "verbalized_confidence"]

</div>

## 📊 What You'll Do in This Demo

<div style="display: flex; margin-bottom: 15px; align-items: center">
  <div style="background-color: #34a853; color: white; border-radius: 50%; width: 30px; height: 30px; display: flex; justify-content: center; align-items: center; margin-right: 15px; flex-shrink: 0"><strong>1</strong></div>
  <div>
    <p style="margin: 0; font-weight: bold"><a href=#section1>Set up LLM and prompts.</a></p>
    <p style="margin: 0; color: rgba(95, 99, 104, 0.8)">Set up LLM instance and load example data prompts.</p>
  </div>
</div>

<div style="display: flex; margin-bottom: 15px; align-items: center">
  <div style="background-color: #34a853; color: white; border-radius: 50%; width: 30px; height: 30px; display: flex; justify-content: center; align-items: center; margin-right: 15px; flex-shrink: 0"><strong>2</strong></div>
  <div>
    <p style="margin: 0; font-weight: bold"><a href=#section2>Generate LLM Responses and Confidence Scores</a></p>
    <p style="margin: 0; color: rgba(95, 99, 104, 0.8)">Generate and score LLM responses to the example questions using the <code>CodeGenUQ()</code> class.</p>
  </div>
</div>

<div style="display: flex; margin-bottom: 25px; align-items: center">
  <div style="background-color: #34a853; color: white; border-radius: 50%; width: 30px; height: 30px; display: flex; justify-content: center; align-items: center; margin-right: 15px; flex-shrink: 0"><strong>3</strong></div>
  <div>
    <p style="margin: 0; font-weight: bold"><a href=#section3>Evaluate Hallucination Detection Performance</a></p>
    <p style="margin: 0; color: rgba(95, 99, 104, 0.8)">Visualize model accuracy at different thresholds of the various code-gen UQ confidence scores. Compute precision, recall, and F1-score of hallucination detection.</p>
  </div>
</div>

## ⚖️ Advantages & Limitations

<div style="display: flex; gap: 20px">
  <div style="flex: 1; background-color: rgba(0, 200, 0, 0.1); padding: 15px; border-radius: 8px; border: 1px solid rgba(0, 200, 0, 0.2)">
    <h3 style="color: #2e8b57; margin-top: 0">Pros</h3>
    <ul style="margin-bottom: 0">
      <li><strong>Flexible:</strong> Works with various Black-box and White-box scorers</li>
    </ul>
  </div>
  
  <div style="flex: 1; background-color: rgba(200, 0, 0, 0.1); padding: 15px; border-radius: 8px; border: 1px solid rgba(200, 0, 0, 0.2)">
    <h3 style="color: #b22222; margin-top: 0">Cons</h3>
    <ul style="margin-bottom: 0">
      <li><strong>Higher Cost:</strong> Requires multiple generations per prompt</li>
      <li><strong>Slower:</strong> Multiple and long generations and comparisons increase latency</li>
    </ul>
  </div>
</div>

<a id='section1'></a>
## 1. Set up LLM and Prompts

In [ ]:
import pandas as pd
from uqlm.utils.dataloader import load_example_dataset
from uqlm.utils.prompts.codegen import python_prompt_template, python_prompt_template_stdio
from uqlm.utils.code_evaluation import evaluate_python_code

In [7]:
df = load_example_dataset("livecodebench", n=5)

Using the latest cached version of the dataset since livecodebench/code_generation_lite couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'release_latest' at /Users/c594266/.cache/huggingface/datasets/livecodebench___code_generation_lite/release_latest/0.0.0/4c038560f391c4c05fdf7fd7ae61ae0e6dbd8672f8fe5b95597b78a8dc40a417 (last modified on Thu Feb  5 10:23:28 2026).


Loading dataset - livecodebench...
Processing dataset...
Dataset ready!


In [8]:
df

,question_title,question_content,platform,question_id,starter_code,public_test_cases,metadata,difficulty
0,A. Short Sort,There are three cards with letters $\texttt{a}...,codeforces,1873_A,,"[{""input"": ""6\nabc\nacb\nbac\nbca\ncab\ncba\n""...",{},easy
1,B. Good Kid,Slavic is preparing a present for a friend's b...,codeforces,1873_B,,"[{""input"": ""4\n4\n2 2 1 2\n3\n0 1 2\n5\n4 3 2 ...",{},easy
2,D. 1D Eraser,You are given a strip of paper $s$ that is $n$...,codeforces,1873_D,,"[{""input"": ""8\n6 3\nWBWWWB\n7 3\nWWBWBWW\n5 4\...",{},easy
3,B. Chemistry,"You are given a string $s$ of length $n$, cons...",codeforces,1883_B,,"[{""input"": ""14\n1 0\na\n2 0\nab\n2 1\nba\n3 1\...",{},medium
4,C. Raspberries,"You are given an array of integers $a_1, a_2, ...",codeforces,1883_C,,"[{""input"": ""15\n2 5\n7 3\n3 3\n7 4 1\n5 2\n9 7...",{},medium


In [9]:
prompts = []
for i in range(len(df)):
    if df["platform"].iloc[i] == "leetcode":
        prompt = python_prompt_template(df["question_content"].iloc[i], df["starter_code"].iloc[i])
    else:
        prompt = python_prompt_template_stdio(df["question_content"].iloc[i])
    prompts.append(prompt)

In this example, we use `AzureChatOpenAI` to instantiate our LLM, but any [LangChain Chat Model](https://js.langchain.com/docs/integrations/chat/) may be used. Be sure to **replace with your LLM of choice.**

In [11]:
from dotenv import load_dotenv, find_dotenv
from langchain_openai import AzureChatOpenAI

load_dotenv(find_dotenv())
llm = AzureChatOpenAI(deployment_name="gpt-4o-mini", openai_api_type="azure", openai_api_version="2024-02-15-preview")

<a id='section2'></a>
## 2. Generate LLM Responses and Confidence Scores

### `CodeGenUQ()` - Generate LLM responses for Code generation tasks and compute confidence scores for each response.

<!-- ![Sample Image](https://raw.githubusercontent.com/cvs-health/uqlm/develop/assets/images/black_box_graphic.png) -->

#### 📋 Class Attributes

<table style="border-collapse: collapse; width: 100%; border: 1px solid rgba(127, 127, 127, 0.2);">
  <tr>
    <th style="background-color: rgba(200, 200, 200, 0.2); width: 20%; padding: 8px; text-align: left; border: 1px solid rgba(127, 127, 127, 0.2);">Parameter</th>
    <th style="background-color: rgba(200, 200, 200, 0.2); width: 25%; padding: 8px; text-align: left; border: 1px solid rgba(127, 127, 127, 0.2);">Type & Default</th>
    <th style="background-color: rgba(200, 200, 200, 0.2); width: 55%; padding: 8px; text-align: left; border: 1px solid rgba(127, 127, 127, 0.2);">Description</th>
  </tr>
  <tr>
    <td style="font-weight: bold; padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">llm</td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">BaseChatModel<br><code>default=None</code></td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">A langchain llm `BaseChatModel`. User is responsible for specifying temperature and other relevant parameters to the constructor of the provided `llm` object.</td>
  </tr>
  <tr>
    <td style="font-weight: bold; padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">scorers</td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">List[str]<br><code>default=None</code></td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">Specifies which black box (consistency) scorers to include. Must be subset of ['sequence_probability', 'min_probability', 'mean_token_negentropy', 'min_token_negentropy', 'probability_margin', 'p_true', 'consistency_and_confidence', 'monte_carlo_probability', 'codebleu', 'code_equivalence', 'verbalized_confidence', 'functional_entropy', 'semantic_sets', 'cosine_sim']. If None, defaults to all scorers.
  </tr>    
  <tr>
    <td style="font-weight: bold; padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">device</td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">str or torch.device<br><code>default=None</code></td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">Specifies the device that NLI model use for prediction. Only applies to 'semantic_negentropy', 'semantic_sets_confidence','noncontradiction', 'entailment' scorers. If None, detects and returns the best available PyTorch device. Prioritizes CUDA (NVIDIA GPU), then MPS (macOS), then CPU.</td>
  </tr>
  <tr>
    <td style="font-weight: bold; padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">use_best</td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">bool<br><code>default=True</code></td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">Specifies whether to swap the original response for the uncertainty-minimized response among all sampled responses based on semantic entropy clusters. Only used if `scorers` includes 'semantic_negentropy' or 'noncontradiction'.</td>
  </tr>
  <tr>
    <td style="font-weight: bold; padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">system_prompt</td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">str or None<br><code>default="You are a helpful assistant."</code></td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">Optional argument for user to provide custom system prompt for the LLM.</td>
  </tr>
  <tr>
    <td style="font-weight: bold; padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">max_calls_per_min</td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">int<br><code>default=None</code></td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">Specifies how many API calls to make per minute to avoid rate limit errors. By default, no limit is specified.</td>
  </tr>
  <tr>
    <td style="font-weight: bold; padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">use_n_param</td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">bool<br><code>default=False</code></td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">Specifies whether to use <code>n</code> parameter for <code>BaseChatModel</code>. Not compatible with all <code>BaseChatModel</code> classes. If used, it speeds up the generation process substantially when <code>num_responses</code> is large.</td>
  </tr>
  <tr>
    <td style="font-weight: bold; padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">postprocessor</td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">callable<br><code>default=None</code></td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">A user-defined function that takes a string input and returns a string. Used for postprocessing outputs.</td>
  </tr>
  <tr>
    <td style="font-weight: bold; padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">sampling_temperature</td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">float<br><code>default=1</code></td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">The 'temperature' parameter for LLM to use when generating sampled LLM responses. Must be greater than 0.</td>
  </tr>
  <tr>
    <td style="font-weight: bold; padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">sentence_transformer</td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">str<br><code>default="sentence-transformers/all-MiniLM-L6-v2"</code></td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">Specifies which huggingface sentence transformer to use when computing cosine similarity for consistency_and_confidence. See https://huggingface.co/sentence-transformers?sort_models=likes#models for more information. The recommended sentence transformer is 'sentence-transformers/all-MiniLM-L6-v2'.</td>
  </tr>  
  <tr>
    <td style="font-weight: bold; padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">nli_model_name</td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">str<br><code>default="microsoft/deberta-large-mnli"</code></td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">Specifies which NLI model to use. Must be acceptable input to <code>AutoTokenizer.from_pretrained()</code> and <code>AutoModelForSequenceClassification.from_pretrained()</code>.</td>
  </tr>
  <tr>
    <td style="font-weight: bold; padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">max_length</td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">int<br><code>default=2000</code></td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">Specifies the maximum allowed string length for LLM responses for NLI computation. Responses longer than this value will be truncated in NLI computations to avoid <code>OutOfMemoryError</code>.</td>
  </tr>
  <tr>
    <td style="font-weight: bold; padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">return_responses</td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">str<br><code>default="all"</code></td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">If a postprocessor is used, specifies whether to return only postprocessed responses, only raw responses, or both. Specified with 'postprocessed', 'raw', or 'all', respectively.</td>
  </tr>
</table>

#### 🔍 Parameter Groups

<div style="display: flex; gap: 20px; margin-bottom: 20px">
  <div style="flex: 1; padding: 10px; background-color: rgba(0, 100, 200, 0.1); border-radius: 5px; border: 1px solid rgba(0, 100, 200, 0.2);">
    <p style="font-weight: bold">🧠 LLM-Specific</p>
    <ul>
      <li><code>llm</code></li>
      <li><code>system_prompt</code></li>
      <li><code>sampling_temperature</code></li>
    </ul>
  </div>
  <div style="flex: 1; padding: 10px; background-color: rgba(0, 200, 0, 0.1); border-radius: 5px; border: 1px solid rgba(0, 200, 0, 0.2);">
    <p style="font-weight: bold">📊 Confidence Scores</p>
    <ul>
      <li><code>scorers</code></li>
      <li><code>use_best</code></li>
      <li><code>nli_model_name</code></li>
      <li><code>sentence_transformer</code></li>
      <li><code>postprocessor</code></li>
    </ul>
  </div>
  <div style="flex: 1; padding: 10px; background-color: rgba(200, 150, 0, 0.1); border-radius: 5px; border: 1px solid rgba(200, 150, 0, 0.2);">
    <p style="font-weight: bold">🖥️ Hardware</p>
    <ul>
      <li><code>device</code></li>
    </ul>
  </div>
  <div style="flex: 1; padding: 10px; background-color: rgba(200, 0, 200, 0.1); border-radius: 5px; border: 1px solid rgba(200, 0, 200, 0.2);">
    <p style="font-weight: bold">⚡ Performance</p>
    <ul>
      <li><code>max_calls_per_min</code></li>
      <li><code>use_n_param</code></li>
    </ul>
  </div>
</div>

#### 💻 Usage Examples

```python
# Basic usage with default parameters
cguq = CodeGenUQ(llm=llm)

# Using GPU acceleration, default scorers
cguq = CodeGenUQ(llm=llm, device=torch.device("cuda"))

# Custom scorer list
cguq = CodeGenUQ(llm=llm, scorers=["semantic_negentropy", "exact_match", "cosine_sim"])

# High-throughput configuration with rate limiting
cguq = CodeGenUQ(llm=llm, max_calls_per_min=200, use_n_param=True) 
```

In [12]:
scorers = ["sequence_probability", "min_probability", "mean_token_negentropy", "min_token_negentropy", "probability_margin", "p_true", "monte_carlo_probability", "codebleu", "code_equivalence", "verbalized_confidence", "functional_entropy", "semantic_sets", "cosine_sim"]

In [13]:
from uqlm import CodeGenUQ

cg = CodeGenUQ(llm=llm, max_calls_per_min=150, language="python", scorers=scorers)

/Users/c594266/uqlm/uqlm/scorers/shortform/white_box.py:227: UQLMBetaWarning: Scorers based on top_logprobs ('mean_token_negentropy','min_token_negentropy','probability_margin') is in beta. Please use with caution as it may change in future releases.
  beta_warning("Scorers based on top_logprobs ('mean_token_negentropy','min_token_negentropy','probability_margin') is in beta. Please use with caution as it may change in future releases.")
/Users/c594266/.cache/huggingface/modules/transformers_modules/jinaai/jina_hyphen_bert_hyphen_v2_hyphen_qk_hyphen_post_hyphen_norm/3baf9e3ac750e76e8edd3019170176884695fb94/configuration_bert.py:29: UserWarning: optimum is not installed. To use OnnxConfig and BertOnnxConfig, make sure that `optimum` package is installed
  warnings.warn("optimum is not installed. To use OnnxConfig and BertOnnxConfig, make sure that `optimum` package is installed")


### 🔄 Class Methods

<table style="border-collapse: collapse; width: 100%; border: 1px solid rgba(127, 127, 127, 0.2);">
  <tr>
    <th style="background-color: rgba(200, 200, 200, 0.2); width: 25%; padding: 8px; text-align: left; border: 1px solid rgba(127, 127, 127, 0.2);">Method</th>
    <th style="background-color: rgba(200, 200, 200, 0.2); width: 75%; padding: 8px; text-align: left; border: 1px solid rgba(127, 127, 127, 0.2);">Description & Parameters</th>
  </tr>
  <tr>
    <td style="font-weight: bold; vertical-align: top; padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">CodeGenUQ.generate_and_score</td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">
      <p>Generate LLM responses, sampled LLM (candidate) responses, and compute confidence scores for the provided prompts.</p>
      <p><strong>Parameters:</strong></p>
      <ul>
        <li><code>prompts</code> - (<strong>List[str] or List[List[BaseMessage]]</strong>) A list of input prompts for the model.</li>
        <li><code>num_responses</code> - (<strong>int, default=5</strong>) The number of sampled responses used to compute consistency.</li>
        <li><code>show_progress_bars</code> - (<strong>bool, default=True</strong>) If True, displays a progress bar while generating and scoring responses.</li>        
      </ul>
      <p><strong>Returns:</strong> <code>UQResult</code> containing data (prompts, responses, sampled responses, and confidence scores) and metadata</p>
      <div style="background-color: rgba(0, 200, 0, 0.1); padding: 8px; border-radius: 3px; margin-top: 10px; border: 1px solid rgba(0, 200, 0, 0.2); margin-right: 5px; box-sizing: border-box; width: 100%;">
        <strong>💡 Best For:</strong> Complete end-to-end uncertainty quantification when starting with prompts.
      </div>
    </td>
  </tr>
  <tr>
    <td style="font-weight: bold; vertical-align: top; padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">CodeGenUQ.score</td>
    <td style="padding: 8px; border: 1px solid rgba(127, 127, 127, 0.2);">
      <p>Compute confidence scores on provided LLM responses. Should only be used if responses and sampled responses are already generated.</p>
      <p><strong>Parameters:</strong></p>
      <ul>
        <li><code>responses</code> - (<strong>List[str]</strong>) A list of LLM responses for the prompts.</li>
        <li><code>sampled_responses</code> - (<strong>List[List[str]]</strong>) A list of lists of sampled LLM responses for each prompt. Used to compute consistency scores by comparing to the corresponding response from <code>responses</code>.</li>
        <li><code>show_progress_bars</code> - (<strong>bool, default=True</strong>) If True, displays a progress bar while scoring responses.</li>  
      </ul>
      <p><strong>Returns:</strong> <code>UQResult</code> containing data (responses, sampled responses, and confidence scores) and metadata</p>
      <div style="background-color: rgba(0, 200, 0, 0.1); padding: 8px; border-radius: 3px; margin-top: 10px; border: 1px solid rgba(0, 200, 0, 0.2); margin-right: 5px; box-sizing: border-box; width: 100%;">
        <strong>💡 Best For:</strong> Computing uncertainty scores when responses are already generated elsewhere.
      </div>
    </td>
  </tr>
</table>

In [14]:
results = await cg.generate_and_score(prompts=prompts, num_responses=5)
df_results = results.to_df()
df_results.head()

Output()

,prompt,response,logprob,sampled_responses,sampled_logprob,verbalized_confidence,cosine_sim,cosine_sim_pair_score,sequence_probability,min_probability,...,code_bleu_pair_score,semantic_entropy,tokenprob_semantic_entropy,num_semantic_set,semantic_negentropy,semantic_negentropy_whitebox,semantic_sets_confidence,cluster_indice,equivalence_rate,original_equivalence_score
0,You are an expert Python programmer. You alway...,import sys\n\ninput = sys.stdin.read\ndata = i...,"[{'token': 'import', 'bytes': [105, 109, 112, ...",[import sys\n\ninput = sys.stdin.read\ndata = ...,"[[{'token': 'import', 'bytes': [105, 109, 112,...",0.8,0.957124,"[0.951094776391983, 0.9341085255146027, 0.9611...",8.304889e-01,0.005024,...,"[0.5793904931000395, 0.42504134644363456, 0.51...",0.515804,0.515612,3,0.515804,0.515612,0.6,"[[0, 1, 2, 4], [3], [5]]",0.6,"[1.0, 1.0, 0.0, 1.0, 0.0]"
1,You are an expert Python programmer. You alway...,import sys\nfrom functools import reduce\n\nin...,"[{'token': 'import', 'bytes': [105, 109, 112, ...",[import sys\nfrom functools import reduce\n\ni...,"[[{'token': 'import', 'bytes': [105, 109, 112,...",0.8,0.987919,"[0.9992497265338898, 0.9939277172088623, 0.977...",9.017918e-01,0.028097,...,"[0.7433975761415965, 0.582089187813875, 0.5385...",0.748537,0.741516,2,0.748537,0.741516,0.8,"[[0, 1, 2, 3, 4], [5]]",0.8,"[1.0, 1.0, 1.0, 1.0, 0.0]"
2,You are an expert Python programmer. You alway...,import sys\ninput = sys.stdin.read\n\ndef mini...,"[{'token': 'import', 'bytes': [105, 109, 112, ...",[import sys\n\ninput = sys.stdin.read\ndata = ...,"[[{'token': 'import', 'bytes': [105, 109, 112,...",1.0,0.958039,"[0.9519636034965515, 0.9262432157993317, 0.939...",8.864005e-01,0.091721,...,"[0.5030152921779345, 0.45252791360487976, 0.43...",1.000000,1.000000,1,1.000000,1.000000,1.0,"[[0, 1, 2, 3, 4, 5]]",1.0,"[1.0, 1.0, 1.0, 1.0, 1.0]"
3,You are an expert Python programmer. You alway...,import sys\nfrom collections import Counter\n\...,"[{'token': 'import', 'bytes': [105, 109, 112, ...",[import sys\nfrom collections import Counter\n...,"[[{'token': 'import', 'bytes': [105, 109, 112,...",1.0,0.984505,"[0.9777904450893402, 0.9887286126613617, 0.989...",7.766716e-01,0.004603,...,"[0.5294873917007472, 0.5614894928431385, 0.550...",0.748537,0.684238,2,0.748537,0.684238,0.8,"[[0], [1, 2, 3, 4, 5]]",0.0,"[0.0, 0.0, 0.0, 0.0, 0.0]"
4,You are an expert Python programmer. You alway...,import sys\nfrom collections import defaultdic...,"[{'token': 'import', 'bytes': [105, 109, 112, ...",[import sys\nfrom collections import defaultdi...,"[[{'token': 'import', 'bytes': [105, 109, 112,...",0.8,0.927674,"[0.9409986734390259, 0.9128977358341217, 0.940...",4.558673e-19,0.000000,...,"[0.21403742640587825, 0.21010027177530238, 0.2...",0.000000,0.387569,6,0.000000,0.387569,0.0,"[[0], [1], [2], [3], [4], [5]]",0.0,"[0.0, 0.0, 0.0, 0.0, 0.0]"


In [15]:
df_results = pd.concat([df_results, df.loc[df_results.index]], axis=1)

In [13]:
df_results.head()

,prompt,response,logprob,sampled_responses,sampled_logprob,verbalized_confidence,cosine_sim,cosine_sim_pair_score,min_probability,sequence_probability,...,question_content,platform,question_id,contest_id,contest_date,starter_code,difficulty,public_test_cases,private_test_cases,metadata
0,You are an expert Python programmer. You alway...,```python\nimport sys\n\ninput = sys.stdin.rea...,"[{'token': '```', 'bytes': [96, 96, 96], 'logp...",[t = int(input())\nresults = []\nfor _ in rang...,"[[{'token': 't', 'bytes': [116], 'logprob': -1...",1.0,0.944298,"[0.9079413712024689, 0.9345949292182922, 0.995...",0.018118,8.800432e-01,...,There are three cards with letters $\texttt{a}...,codeforces,1873_A,1873,2023-08-21T00:00:00,,easy,"[{""input"": ""6\nabc\nacb\nbac\nbca\ncab\ncba\n""...",eJxrYJmaz8gABhEZQEZ0tVJmXkFpiZKVgpJhTF5iUnJMnp...,{}
1,You are an expert Python programmer. You alway...,import sys\nfrom functools import reduce\n\nin...,"[{'token': 'import', 'bytes': [105, 109, 112, ...",[import sys\nimport math\n\ninput = sys.stdin....,"[[{'token': 'import', 'bytes': [105, 109, 112,...",1.0,0.980687,"[0.9575135409832001, 0.9802885353565216, 0.988...",0.075829,9.349250e-01,...,Slavic is preparing a present for a friend's b...,codeforces,1873_B,1873,2023-08-21T00:00:00,,easy,"[{""input"": ""4\n4\n2 2 1 2\n3\n0 1 2\n5\n4 3 2 ...",eJyUvb2uNVvSlIuBi8UNDLWNUf9VgytB4jXBwOmDRGMcHR...,{}
2,You are an expert Python programmer. You alway...,import sys\ninput = sys.stdin.read\n\ndef min_...,"[{'token': 'import', 'bytes': [105, 109, 112, ...",[import sys\ninput = sys.stdin.read\n\ndef min...,"[[{'token': 'import', 'bytes': [105, 109, 112,...",1.0,0.952506,"[0.9691091775894165, 0.9331307113170624, 0.959...",0.005703,8.407014e-01,...,You are given a strip of paper $s$ that is $n$...,codeforces,1873_D,1873,2023-08-21T00:00:00,,easy,"[{""input"": ""8\n6 3\nWBWWWB\n7 3\nWWBWBWW\n5 4\...",eJyUvb2SY8vWXSdDplz6O65No/APyMyXSEaoTNGgU2IELw...,{}
3,You are an expert Python programmer. You alway...,import sys\nfrom collections import Counter\n\...,"[{'token': 'import', 'bytes': [105, 109, 112, ...",[import sys\nfrom collections import Counter\n...,"[[{'token': 'import', 'bytes': [105, 109, 112,...",0.8,0.965058,"[0.9933294057846069, 0.9738149046897888, 0.975...",0.021135,8.727980e-01,...,"You are given a string $s$ of length $n$, cons...",codeforces,1883_B,1883,2023-09-22T00:00:00,,medium,"[{""input"": ""14\n1 0\na\n2 0\nab\n2 1\nba\n3 1\...",eJylkUELwiAYhiP6FZ3M84i5Glvdu9ahS5EdpnObG9hg7h...,{}
4,You are an expert Python programmer. You alway...,import sys\nfrom collections import defaultdic...,"[{'token': 'import', 'bytes': [105, 109, 112, ...",[import sys\nfrom collections import Counter\n...,"[[{'token': 'import', 'bytes': [105, 109, 112,...",0.8,0.929624,"[0.8982242345809937, 0.9368771314620972, 0.947...",0.000000,7.055932e-17,...,"You are given an array of integers $a_1, a_2, ...",codeforces,1883_C,1883,2023-09-22T00:00:00,,medium,"[{""input"": ""15\n2 5\n7 3\n3 3\n7 4 1\n5 2\n9 7...",eJyc3TuOLluWJGYKHAaFhZJbaH9ud46EAFNkC60kG+hqgS...,{}


In [16]:
# 3. Evaluate Responses
df_results = evaluate_python_code(df_results)

In [17]:
df_results.head()

,prompt,response,logprob,sampled_responses,sampled_logprob,verbalized_confidence,cosine_sim,cosine_sim_pair_score,sequence_probability,min_probability,...,question_content,platform,question_id,starter_code,public_test_cases,metadata,difficulty,unit_test_passed,stderr,stdout
0,You are an expert Python programmer. You alway...,import sys\n\ninput = sys.stdin.read\ndata = i...,"[{'token': 'import', 'bytes': [105, 109, 112, ...",[import sys\n\ninput = sys.stdin.read\ndata = ...,"[[{'token': 'import', 'bytes': [105, 109, 112,...",0.8,0.957124,"[0.951094776391983, 0.9341085255146027, 0.9611...",8.304889e-01,0.005024,...,There are three cards with letters $\texttt{a}...,codeforces,1873_A,,"[{'input': '6 abc acb bac bca cab cba ', 'outp...",{},easy,0,Wrong answer at output_line_idx=4: YES != NO,
1,You are an expert Python programmer. You alway...,import sys\nfrom functools import reduce\n\nin...,"[{'token': 'import', 'bytes': [105, 109, 112, ...",[import sys\nfrom functools import reduce\n\ni...,"[[{'token': 'import', 'bytes': [105, 109, 112,...",0.8,0.987919,"[0.9992497265338898, 0.9939277172088623, 0.977...",9.017918e-01,0.028097,...,Slavic is preparing a present for a friend's b...,codeforces,1873_B,,[{'input': '4 4 2 2 1 2 3 0 1 2 5 4 3 2 3 4 9 ...,{},easy,1,,
2,You are an expert Python programmer. You alway...,import sys\ninput = sys.stdin.read\n\ndef mini...,"[{'token': 'import', 'bytes': [105, 109, 112, ...",[import sys\n\ninput = sys.stdin.read\ndata = ...,"[[{'token': 'import', 'bytes': [105, 109, 112,...",1.0,0.958039,"[0.9519636034965515, 0.9262432157993317, 0.939...",8.864005e-01,0.091721,...,You are given a strip of paper $s$ that is $n$...,codeforces,1873_D,,[{'input': '8 6 3 WBWWWB 7 3 WWBWBWW 5 4 BWBWB...,{},easy,1,,
3,You are an expert Python programmer. You alway...,import sys\nfrom collections import Counter\n\...,"[{'token': 'import', 'bytes': [105, 109, 112, ...",[import sys\nfrom collections import Counter\n...,"[[{'token': 'import', 'bytes': [105, 109, 112,...",1.0,0.984505,"[0.9777904450893402, 0.9887286126613617, 0.989...",7.766716e-01,0.004603,...,"You are given a string $s$ of length $n$, cons...",codeforces,1883_B,,[{'input': '14 1 0 a 2 0 ab 2 1 ba 3 1 abb 3 2...,{},medium,0,Wrong answer at output_line_idx=1: YES != NO,
4,You are an expert Python programmer. You alway...,import sys\nfrom collections import defaultdic...,"[{'token': 'import', 'bytes': [105, 109, 112, ...",[import sys\nfrom collections import defaultdi...,"[[{'token': 'import', 'bytes': [105, 109, 112,...",0.8,0.927674,"[0.9409986734390259, 0.9128977358341217, 0.940...",4.558673e-19,0.000000,...,"You are given an array of integers $a_1, a_2, ...",codeforces,1883_C,,[{'input': '15 2 5 7 3 3 3 7 4 1 5 2 9 7 7 3 9...,{},medium,0,Wrong answer at output_line_idx=1: 0 != 2,
